# Notebook 01 — The Health Records: Who, Where, and When
**GeoAI for Climate, Environment and Health**

---

> **Where you are in the pipeline**
>
> ```
> [📋 Health records] → 🛰 Satellite → 🌍 Environment → 🔗 Link → ⏱ Timing → 🎯 Results & GeoAI
> ```
>
> This notebook is about the **people**. Before we can ask what climate does to health,
> we need a dataset of individuals — each with a location, a time, and an outcome.
> We use the Zimbabwe Demographic and Health Survey (DHS) birth records for exactly this.

---

---
## The Landscape of Spatial Health Data

Health research uses several types of georeferenced data, and the spatial unit you work with shapes every downstream analysis — what exposures can be linked, what biases arise, and what inferences are possible.

| Spatial unit | Typical source | Geographic reference | Best suited for |
|---|---|---|---|
| **Health facility** | MoH registry, HMIS | GPS coordinate of the clinic or hospital | Access analysis, catchment delineation, service-quality studies |
| **DHS cluster** | DHS Programme household survey | Cluster centroid, displaced 0–5 km for privacy | Individual-level outcomes from a probability sample |
| **Administrative area** | National census, national GIS | District or ward polygon | Population surveillance, routine aggregate reporting |
| **Individual household** | HDSS, community health | Exact GPS (rare in public datasets) | Precision exposure mapping; strict privacy controls required |

**Why does the spatial unit matter for environmental health research?**

- A **facility record** only captures care-seekers, not the whole population, creating a selection bias. Useful for access studies but cannot give a true population birth rate.
- A **district aggregate** covers everyone but erases within-district environmental variation. A 5,000 km² district averages over steep temperature or rainfall gradients that matter for health.
- A **DHS cluster** gives individual birth records from a probability sample — enabling individual-level regression with confounders (wealth, education, parity) — but coordinates are displaced up to 5 km (rural) to protect participant privacy.

**In this course** we use DHS cluster data because it provides individual birth records at sub-district spatial resolution. The spatial join methods you learn in Notebooks 04 and 05 apply equally to linking satellite exposures to facility coordinates, ward centroids, or household GPS points.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.cm import get_cmap

import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")


---
## 1. Load and Inspect the Birth Records

The DHS Programme collects nationally representative household survey data across low- and middle-income countries. Birth histories are collected retrospectively from women aged 15–49. This dataset covers births in the five years prior to the 2015 Zimbabwe DHS fieldwork (birth years 2011–2015).

**Key design features to keep in mind throughout this workshop:**

- Sampling is **two-stage**: clusters (enumeration areas) are drawn first, then households within clusters. This means records within the same cluster are not independent.
- Each record carries a `sample_weight` that corrects for differential sampling probabilities and non-response. Unweighted statistics are biased toward over-sampled strata (typically urban areas).
- GPS coordinates are recorded at the **cluster centroid**, not the household, and are then deliberately displaced — see Section 3.

In [ ]:
births = pd.read_csv(DATA_DIR / 'zw_births_raw.csv')

print(f"Shape: {births.shape[0]:,} rows × {births.shape[1]} columns")
print(f"Birth years: {births['birth_year'].min()}–{births['birth_year'].max()}")
print(f"Provinces ({births['province'].nunique()}): {sorted(births['province'].unique())}")
print()

# Missing data is an important feature of DHS — not every outcome is recorded for every birth
outcome_cols = ['birth_weight_g', 'low_birth_weight', 'neonatal_death',
                'small_at_birth', 'preterm', 'anc_4plus', 'facility_delivery']
missing = births[outcome_cols].isna().sum()
missing_pct = (missing / len(births) * 100).round(1)
print("Missing values in key outcome columns:")
print(pd.DataFrame({'n_missing': missing, 'pct_missing': missing_pct}).to_string())

In [ ]:
# Records by birth year and province
print("Records by birth year:")
print(births['birth_year'].value_counts().sort_index().to_string())
print()
print("Records by province:")
print(births['province'].value_counts().sort_index().to_string())

**Note on date encoding:** DHS raw data encodes dates as CMC (century month codes). The columns `birth_year`, `birth_month`, `conception_year`, `conception_month` have already been decoded to calendar values. This is non-trivial in original DHS files — CMC counts months from January 1900, so CMC 1368 = December 2013.

**Note on sentinel values in `birth_weight_g`:** DHS raw files use numeric codes ≥ 9000 (9996 = "not weighed at birth", 9998 = "don't know") for missing birth weight. These are *not* real weights and must be excluded before any birth weight analysis. The `low_birth_weight` flag is correctly coded to 0 for these records — a silent trap that inflates apparent birth weight means if you forget to filter. Always restrict birth weight analyses to `birth_weight_g < 9000`.

---
## 2. Exercise 1 — Low Birth Weight Rate by Province (3 points)

Low birth weight (LBW) is defined as birth weight < 2,500 g. Not all births have a recorded weight — many home deliveries are never weighed. DHS encodes these as sentinel values ≥ 9000 g (see note above).

**Task:** Restrict to births with a real recorded weight (`birth_weight_g` not null **and** `birth_weight_g < 9000`). Compute the LBW rate (as a percentage) by province. Sort the result descending by LBW rate. Include a horizontal bar chart.

**Store your results as:**
- `lbw_by_province` — a DataFrame with columns `province`, `lbw_rate_pct`, `n_with_weight`

In [ ]:
# Exclude sentinel codes ≥ 9000 in addition to NA — these are "not weighed" placeholders
weighed = births[(births['birth_weight_g'].notna()) & (births['birth_weight_g'] < 9000)].copy()

lbw_by_province = (
    weighed.groupby('province')
    .agg(
        n_with_weight=('birth_weight_g', 'count'),
        n_lbw=('low_birth_weight', 'sum')
    )
    .assign(lbw_rate_pct=lambda d: (d['n_lbw'] / d['n_with_weight'] * 100).round(2))
    .reset_index()[['province', 'lbw_rate_pct', 'n_with_weight']]
    .sort_values('lbw_rate_pct', ascending=False)
    .reset_index(drop=True)
)

print(lbw_by_province.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(lbw_by_province['province'][::-1],
        lbw_by_province['lbw_rate_pct'][::-1],
        color='#c0392b', edgecolor='white', linewidth=0.5)
ax.axvline(lbw_by_province['lbw_rate_pct'].mean(),
           color='#2c3e50', linestyle='--', linewidth=1.2,
           label=f"Mean across provinces ({lbw_by_province['lbw_rate_pct'].mean():.1f}%)")
ax.set_xlabel('LBW rate (%)')
ax.set_title('Low Birth Weight Rate by Province\nZimbabwe DHS 2015 (births with real recorded weight)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 DHS cluster coordinates are intentionally displaced from true household locations. Why?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> GPS accuracy in Zimbabwe is too low to record precise locations</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> To protect household privacy while preserving geographic utility for research</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> To reduce the file size of the dataset</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> To match cluster coordinates to hospital catchment boundaries</span></label></div>
  <div id="nb01q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Displacement (up to 5 km rural, 2 km urban) protects respondent identities while keeping clusters useful for regional spatial analysis.",ID="nb01q1";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## 3. DHS Cluster Mapping and the Jitter Convention

Each DHS birth record carries the GPS coordinates of its *cluster* (enumeration area centroid), not the household. To protect respondent confidentiality, the DHS Programme displaces cluster coordinates using a controlled random offset before public release:

| Stratum | Displacement |
|---------|-------------|
| Urban clusters | 0–2 km in a random direction |
| Rural clusters | 0–5 km (99% of clusters) |
| Rural clusters (1% random selection) | 0–10 km |

Displacement is constrained to remain within the same administrative district (first-order subdivision).

**Implications for this workshop's spatial analysis:**

- Point-in-polygon tests (assigning a cluster to a province) are almost always correct — provinces are large enough that the displacement stays inside. District-level assignment is less reliable near boundaries.
- Extracting environmental values at displaced coordinates introduces *measurement error in the exposure*, not the outcome. This is non-differential misclassification: it attenuates estimated associations toward the null.
- Analyses using buffers of 5+ km encompass the displacement range for 99% of rural clusters, partially mitigating the bias.

We now plot all 399 clusters from the cluster summary file to visualise spatial coverage.

In [ ]:
clusters = pd.read_csv(DATA_DIR / 'zw_clusters.csv')
print(f"Clusters: {len(clusters)}  |  Provinces: {clusters['province'].nunique()}")
print(f"Urban clusters: {(clusters['urban'] == 1).sum()}  |  Rural: {(clusters['urban'] == 0).sum()}")
clusters.head(3)

In [ ]:
import contextily as ctx
from pyproj import Transformer as _T

provinces = sorted(clusters['province'].unique())
cmap = get_cmap('tab10', len(provinces))
province_colour = {p: cmap(i) for i, p in enumerate(provinces)}

# Project to Web Mercator for basemap
_proj = _T.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
_cx, _cy = _proj.transform(clusters['cluster_lon'].values, clusters['cluster_lat'].values)
clusters = clusters.copy()
clusters['_mx'] = _cx; clusters['_my'] = _cy

fig, ax = plt.subplots(figsize=(9, 8))
for province in provinces:
    sub = clusters[clusters['province'] == province]
    ax.scatter(sub['_mx'], sub['_my'],
               c=[province_colour[province]], s=25, alpha=0.85, linewidths=0)

try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=7)
except Exception:
    pass

ax.set_axis_off()
legend_handles = [mpatches.Patch(color=province_colour[p], label=p) for p in provinces]
ax.legend(handles=legend_handles, fontsize=8, loc='lower left',
          title='Province', title_fontsize=9, framealpha=0.9)
ax.set_title('Zimbabwe DHS 2015 — Cluster Locations by Province\n399 clusters, coordinates randomly displaced 0–5 km',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

The Eastern Highlands (high longitude, ~18°S–20°S) are visible as a cluster-sparse region — lower population density yields fewer DHS enumeration areas.  Metropolitan Province appears as a tight cluster of urban points in the southwest.

---
## 4. Exercise 2 — Province with Highest Neonatal Mortality Rate (3 points)

Neonatal mortality is death within the first 28 days of life. The column `neonatal_death` (0/1) records whether each birth resulted in a neonatal death. The rate is conventionally expressed per 1,000 live births.

**Task:** Identify the province with the highest neonatal mortality rate.

**Store:**
- `highest_nm_province` — string: name of the province
- `highest_nm_rate` — float: rate per 1,000 births, rounded to 1 decimal place

In [ ]:
nm_by_province = (
    births.groupby('province')
    .agg(
        n_births=('neonatal_death', 'count'),
        n_deaths=('neonatal_death', 'sum')
    )
    .assign(nm_rate=lambda d: (d['n_deaths'] / d['n_births'] * 1000).round(1))
    .sort_values('nm_rate', ascending=False)
)

highest_nm_province = str(nm_by_province.index[0])
highest_nm_rate = float(nm_by_province['nm_rate'].iloc[0])

print(nm_by_province[['n_births', 'n_deaths', 'nm_rate']].to_string())
print(f"\nHighest neonatal mortality: {highest_nm_province} ({highest_nm_rate} per 1,000 births)")

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 Urban women in the Zimbabwe DHS had substantially higher ANC attendance rates than rural women. What is the most likely explanation?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Urban women are wealthier and have more education on average</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Health facilities are more physically accessible in urban areas</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> The DHS oversampled urban women who attend ANC</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Both A and B — urban advantage reflects wealth, education, and proximity to services</span></label></div>
  <div id="nb01q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="D",E="Urban health advantages are multifactorial: proximity to facilities, higher education, and greater household income all contribute.",ID="nb01q2";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## 5. Urban–Rural Disparities in Birth Outcomes

Zimbabwe's DHS design deliberately over-samples urban clusters to allow urban-specific estimates. The `urban` column (0 = rural, 1 = urban) identifies the DHS stratum.

We compute three indicators as unweighted descriptive statistics. For provincial or national estimates, weights would be required.

In [ ]:
stratum_labels = {0: 'Rural', 1: 'Urban'}

# Exclude sentinel codes before computing birth weight statistics
real_weight = births[(births['birth_weight_g'].notna()) & (births['birth_weight_g'] < 9000)]

bw_by_stratum = (
    real_weight
    .groupby('urban')['birth_weight_g']
    .agg(['mean', 'count'])
    .rename(index=stratum_labels)
    .rename(columns={'mean': 'mean_bw_g', 'count': 'n'})
    .round({'mean_bw_g': 1})
)

# Facility delivery rate
fd_by_stratum = (
    births.dropna(subset=['facility_delivery'])
    .groupby('urban')['facility_delivery']
    .agg(['mean', 'count'])
    .rename(index=stratum_labels)
    .rename(columns={'mean': 'facility_delivery_pct', 'count': 'n'})
    .assign(facility_delivery_pct=lambda d: (d['facility_delivery_pct'] * 100).round(1))
)

# ANC 4+ coverage
anc_by_stratum = (
    births.dropna(subset=['anc_4plus'])
    .groupby('urban')['anc_4plus']
    .agg(['mean', 'count'])
    .rename(index=stratum_labels)
    .rename(columns={'mean': 'anc4plus_pct', 'count': 'n'})
    .assign(anc4plus_pct=lambda d: (d['anc4plus_pct'] * 100).round(1))
)

print("Mean birth weight (g) — real recorded weights only (sentinel codes excluded):")
print(bw_by_stratum.to_string())
print()
print("Facility delivery rate (%):")
print(fd_by_stratum.to_string())
print()
print("ANC 4+ coverage (%):")
print(anc_by_stratum.to_string())

Urban areas show substantially higher facility delivery and ANC 4+ rates. The birth weight gap is smaller and should be interpreted carefully: rural births that *have* a recorded weight are disproportionately facility deliveries, so the rural birth weight sample is already positively selected relative to all rural births.

**A note on weighted vs. unweighted estimates:** The table above is unweighted. Because urban clusters are over-sampled, unweighted national means would over-represent urban women's outcomes. Always apply `sample_weight` when computing national prevalence figures for publication.

---
## 6. Exercise 3 — Birth Weight by ANC Attendance (2 points)

Antenatal care (ANC) is hypothesised to improve birth outcomes through early detection of complications, iron/folate supplementation, and intermittent preventive treatment for malaria. `anc_4plus` (0/1) encodes whether the mother attended at least 4 ANC visits (the WHO-recommended minimum at the time of the 2015 DHS).

**Task:** Among births with a real recorded weight (`birth_weight_g` not null and `birth_weight_g < 9000`) *and* `anc_4plus` not null, compute the mean birth weight (in grams) separately for mothers who attended 4+ ANC visits versus those who attended fewer.

**Store:**
- `mean_bw_anc4plus` — float: mean birth weight (g) for the ANC 4+ group
- `mean_bw_no_anc` — float: mean birth weight (g) for the <4 visits group

In [ ]:
# Exclude sentinel codes and restrict to births where ANC status is known
weighed_anc = births[
    (births['birth_weight_g'].notna()) &
    (births['birth_weight_g'] < 9000) &
    (births['anc_4plus'].notna())
].copy()

mean_bw_anc4plus = float(
    weighed_anc.loc[weighed_anc['anc_4plus'] == 1, 'birth_weight_g'].mean()
)
mean_bw_no_anc = float(
    weighed_anc.loc[weighed_anc['anc_4plus'] == 0, 'birth_weight_g'].mean()
)

print(f"Mean birth weight — ANC 4+  : {mean_bw_anc4plus:.1f} g  (n={(weighed_anc['anc_4plus']==1).sum()})")
print(f"Mean birth weight — <4 ANC  : {mean_bw_no_anc:.1f} g  (n={(weighed_anc['anc_4plus']==0).sum()})")
print(f"Gap                         : {mean_bw_anc4plus - mean_bw_no_anc:+.1f} g")
print()
print("Note: this unadjusted gap conflates the effect of ANC with urban residence,"
      " wealth, and education — all correlated with ANC attendance.")

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 Birth weight is recorded as 9996 g for some records in the raw DHS file. What does this value represent?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q3" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> An exceptionally large baby</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q3" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> A data entry error that should be corrected</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q3" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> A sentinel/flag value meaning the weight was not measured or not known</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q3" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> The combined weight of twins</span></label></div>
  <div id="nb01q3-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="C",E="DHS raw files use numeric codes (9996, 9998, etc.) for missing or unknown values. These must be excluded before any statistical analysis.",ID="nb01q3";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Recap

**Key points from this notebook:**

- **DHS birth records** are retrospective, multi-birth, cluster-sampled data with substantial missing values in outcome columns. Birth weight missingness is not random — it is concentrated in rural, home-delivered births, making it a potentially informative missing-data mechanism.

- **Cluster-level geocoding with jitter** is the defining spatial constraint of public DHS data. Coordinates are displaced 0–5 km (rural) or 0–2 km (urban). This introduces *exposure measurement error* in spatial analyses, biasing exposure–outcome associations toward the null. Larger spatial buffers reduce but do not eliminate this bias.

- **Sample weights** are required for nationally representative estimates. Urban clusters are over-sampled; ignoring weights overstates outcomes that are more prevalent in urban areas.

- **Urban–rural gradients** are strong in Zimbabwe's DHS data: urban women have markedly higher facility delivery and ANC coverage rates. These access differences are major confounders in exposure–outcome analyses.

**In Notebook 02** we explore the satellite and reanalysis climate data that underlies the trimester-level exposure columns (`temp_t1/t2/t3`, `rain_t1/t2/t3`, `ndvi_t1/t2/t3`, `pm25_t1/t2/t3`). Understanding Zimbabwe's seasonal structure is the necessary foundation before interpreting those exposures.

In [ ]:
%%html
<div id="nb01-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 1 &#8212; Spatial Health Data &#8212; Quiz Results</div>
  <div id="nb01-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb01-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:'nb01q1',correct:'B'},{id:'nb01q2',correct:'D'},{id:'nb01q3',correct:'C'}];
  var PASS=67;
  var NEXT='02_earth_engine_extraction.ipynb';
  var NEXT_LABEL='Notebook 2: Earth Engine &amp; Satellite Data';
  var NBID='nb01';
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem('mcq_'+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+'-score-display');
    var act=document.getElementById(NBID+'-next-action');
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions above to see your score. ('+(total-answered)+' remaining)</p>';
      if(act)act.innerHTML='';
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; MODULE PASSED</b><br><small>Your progress is saved &#8212; you can close and return to this notebook.</small></div>';
      if(act){
        localStorage.setItem(NBID+'_passed','true');
        if(NEXT){act.innerHTML='<a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px;margin-top:8px">&#10145; Continue to '+NEXT_LABEL+'</a>';}
        else{act.innerHTML='<div style="background:#cfe2ff;color:#084298;padding:12px 16px;border-radius:8px;margin-top:8px"><b>&#127881; You have completed the full GeoAI pipeline course! Congratulations.</b></div>';}
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer, then your score updates automatically.</small></div>';
      if(act)act.innerHTML='';
    }
  }
  document.addEventListener('mcq-update',update);
  if(document.readyState==='loading')document.addEventListener('DOMContentLoaded',update);
  else update();
  setInterval(update,3000);
})();
</script>

---

## ✅ Notebook 01 complete

You have a clean, geocoded dataset of Zimbabwe DHS birth records — each with a location, a birth outcome, and mother characteristics.

**What is missing?** The environment. But before we explore the climate data, it is worth asking: where does that data actually come from?

➡️ **Notebook 02** introduces Google Earth Engine and the satellite products used to characterise Zimbabwe's climate — before we look at the data itself, we understand how it was made.